In [1]:
from langchain.chat_models import init_chat_model
from open_deep_research.knowledge import (
    dataset_info,
    abbreviation
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_experimental.utilities import PythonREPL
import textwrap
import os
# os.load_dotenv()

from open_deep_research.prompts import (
    report_planner_query_writer_instructions,
    report_planner_instructions,
    query_writer_instructions,
    section_writer_instructions_v2,
    section_writer_instructions,
    final_section_writer_instructions,
    final_section_writer_instructions_v2,
    section_grader_instructions,
    section_writer_inputs,
    visualization_instructions,
    sql_instructions,
    sql_grader_instructions,
    visualization_instructions
)
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field
import operator

sql_interpret_instruction = """You are **Data Insight Agent**, an analytical assistant that turns raw SQL outputs into concise, executive-ready briefings.
<sql_result>
{sql_result}
</sql_result>"""

class SQLResponse(BaseModel):
    sql_script: str = Field(None, description="SQL script if cant retrive output empty string ")
    explaination: str = Field(None, description="explain why cant retrive given data schema, else ouput empty")


class ClarifyQuestion(BaseModel):
    follow_up_questions: List[str] = Field(None, description="Alternative questions to query if there is not enough information in original question")

class SQLInterpret(BaseModel):
    insight: str = Field(None, description="Insight about sql result and question")
    summarization:  str = Field(None, description="Summarization about sql result and question")

class VisualizeScript(BaseModel):
    visualize_script: str = Field(None, description="visualize script if cant visulize output empty string ")


llm_flash = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

# llm_flash = llm_flash.with_structured_output(SQLResponse)
llm_4o = ChatOpenAI(
    model_name="gpt-4o-mini",      # or "gpt-4o-mini" for lower cost / latency
    
    temperature=0.0,
    # Optional:
    # max_tokens=2048,
    # timeout=30,              # seconds
    # base_url="https://api.openai.com/v1",
)
import textwrap
from google.cloud import bigquery

def python_execute(code: str) -> str:
    """
    Execute Python code in a fresh PythonREPL and return stdout / errors.
    """
    import traceback
    import sys

    wrapped = "try:\n"
    wrapped += textwrap.indent(code, "    ")
    wrapped += textwrap.dedent(
        """
        except Exception as e:
            print(f"error: {e}")
    """
    )
    # print(wrapped)
    python_repl = PythonREPL()
    return python_repl.run(wrapped)

def query_bigquery(sql: str, project_id: str = "agentic-ai-463517") -> str:
    """
    Run a BigQuery query in a throw-away PythonREPL sandbox and return its stdout.
    Using `repr(sql)` guarantees the SQL is embedded as a *single* clean string.
    """
    code = f"""
from google.cloud import bigquery
client = bigquery.Client(project={project_id!r})
query_job = client.query({sql!r})

if query_job.result().total_rows == 0:
    print("No results found.")
else:
    print([row[0] for row in query_job.result()])
"""
    return python_execute(code)


# Wrap with the Pydantic schema so every call returns a parsed SQLResponse.
# llm_4o = llm_4o.with_structured_output(SQLResponse)
def sql_writer(llm, question):
    print("sql writer.....")
    project_id = "agentic-ai-463517"
    dataset_name = "ghn_data"
    sql_llm = llm.with_structured_output(SQLResponse)
    clarify_sql_llm = llm.with_structured_output(ClarifyQuestion)
    system_instructions_query = sql_instructions.format(
                        question=question,
                        dataset_info=dataset_info,
                        project_id=project_id,
                        dataset_name=dataset_name,
                        last_error="there was no error in the last query",
                        previous_query="",
                    )
        # print(system_instructions_query)
    result =  sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])
    print("="*100)
    #print(system_instructions_query)
    #print(result.sql_script)
    #print(result.explaination)
    print("="*100)
    
    if result.sql_script != "":
        output = query_bigquery(result.sql_script)
        print(output)
        print("="*100)
        for i in range(3):
            print(result)
            print(output)
            if "error" in output:
                system_instructions_query = sql_instructions.format(
                                question=question,
                                dataset_info=dataset_info,
                                project_id=project_id,
                                dataset_name=dataset_name,
                                last_error=output,
                                previous_query=result.sql_script,
                            )
                print(system_instructions_query)
                result =  sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                                        HumanMessage(content=question)])
                output = query_bigquery(result.sql_script)
                print("="*100)
                #print(result.sql_script)
                #print(result.explaination)
                print(output)
                print("="*100)
            else:
                return output
    else:
        # adding follow up question
        print('Cant write sql')
        print(result.explaination)
        result =  clarify_sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                                        HumanMessage(content=question)])
        print(result.follow_up_questions)

def visualize_data(llm, question, sql_result):
    llm = llm.with_structured_output(VisualizeScript)

    system_instructions_query = visualization_instructions.format(
                        previous_script="",
                        query_result=sql_result,
                        folder_path="./",
                        last_error="",
                    )
        # print(system_instructions_query)
    result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])

    print(result.visualize_script)
    python_execute(result.visualize_script)

def reasoning_data(llm, question, sql_result):
    llm = llm.with_structured_output(SQLInterpret)

    system_instructions_query = sql_interpret_instruction.format(
                        sql_result=sql_result
                    )
        # print(system_instructions_query)
    result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])
    print(result.summarization)
    print(result.insight)





In [2]:
# sql = sql_writer(llm_4o,"What is the main goal of improving NVPTTT productivity by 15% and its expected financial impact?")


In [3]:
# print(
#     """SELECT \n    FORMAT_TIMESTAMP('%Y-%m', so.created_date) as month_year, \n    COUNT(so._id) as total_orders, \n    SUM(CASE WHEN so.status = 'delivered' THEN 1 ELSE 0 END) as successful_deliveries, \n    AVG(TIMESTAMP_DIFF(so.end_success_time, so.created_date, HOUR)) as avg_delivery_time_hours, \n    AVG(CASE WHEN sla.delivery_sla IS NOT NULL THEN TIMESTAMP_DIFF(so.end_success_time, so.created_date, HOUR) - sla.delivery_sla * 24 ELSE NULL END) as avg_sla_difference_hours\nFROM \n    `agentic-ai-463517.ghn_data.shipping_order` as so\nLEFT JOIN \n    `agentic-ai-463517.ghn_data.sla_delivery` as sla \n    ON so.from_district_id = sla.from_district_id \n    AND so.to_district_id = sla.to_district_id\nWHERE \n    so.created_date_partition BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 1 YEAR) AND CURRENT_DATE()\nGROUP BY \n    month_year\nORDER BY \n    month_year DESC"""
# )

In [4]:
query = f"""SELECT DISTINCT warehouse_name
FROM   `agentic-ai-463517.ghn_data.dim_warehouse`
WHERE  warehouse_name IS NOT NULL"""

print(query_bigquery(query))

Python REPL can execute arbitrary code. Use with caution.
/usr/local/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


['M2M Đắk Lắk-10 LQĐ', 'Hà Nam', 'HCM-Quận 1', 'Bưu Cục Thanh Lý Đặc Biệt-HCM', 'Hải Dương', 'Bưu Cục 108 Kim Bài-Thanh Oai-HN', 'Big C Trường Chinh', 'CTO-VinMart+-134ADuong3thang2', 'Quảng Trị', 'Big C Thủ Dầu Một', 'CTO-VinMart+-15AMauThan', 'HCM-Quận 6', 'CTO-VinMart+-1BDinhTienHoang', 'Truck_HN', 'KhoVatTuHN', 'TT GHN', 'Yên Bái', 'HCM-Gò Vấp', 'Kho Lấy Hàng Fulfillment GHN_Hồ Chí Minh', 'Điểm Dịch vụ CircleK', 'Kho Lấy Hàng Thuận An_Bình Dương', 'Thái Nguyên', 'HCM-Phú Nhuận', 'Bưu Cục 693 Điện Biên-TP.Nam Định 02', 'Tuyên Quang', 'BDG-VinMart+-O23-DC01KDCVietSing', 'Bưu Cục 08 Noong Bua-Điện Biên Phủ-Điện Biên 02', 'Bưu Cục Thanh Lý 500K-HCM', 'Bưu Cục 925 Lũy Bán Bích-Q.Tân Phú-HCM', 'Khu vực Hà Nội', 'BDG-VinMart+-62BisCachMangThangTam', 'KV3 Bưu cục 284 Tống Duy Tân - Thanh Hóa', 'Bưu Cục 73 Quang Trung-TP.Tuyên Quang', 'Quảng Ngãi', 'KV2 Bưu cục 106 Hùng Vương - Đông Hà', 'HCM-LuanChuyen-353ANguyenThaiBinh', 'KV1 Bưu cục 970 Hùng Vương - Chư Sê', 'TP.Hòa Bình_HB', 'Trà Vinh'

In [5]:
from open_deep_research.utils import query_bigquery
query="SELECT \n    so.deliver_user, \n    COUNT(so.order_code_hash) AS total_deliveries,\n    SUM(CASE WHEN so.status = 'Delivered' THEN 1 ELSE 0 END) AS successful_deliveries,\n    AVG(TIMESTAMP_DIFF(so.end_delivery_time, so.first_delivered_time, MINUTE)) AS average_delivery_time_minutes,\n    SUM(ro.rev) AS total_revenue,\n    AVG(sd.delivery_sla) AS average_sla_days\nFROM \n    agentic-ai-463517.ghn_data.shipping_order so\nLEFT JOIN \n    agentic-ai-463517.ghn_data.revenue_order ro ON so.order_code_hash = ro.order_code_hash\nLEFT JOIN \n    agentic-ai-463517.ghn_data.sla_delivery sd ON so.to_district_id = sd.to_district_id\nWHERE \n    so.created_date_partition BETWEEN '2023-10-01' AND '2023-10-31'\nGROUP BY \n    so.deliver_user\nORDER BY \n    total_deliveries DESC;"

query_bigquery(query)

/var/folders/tg/jyzhfxz54nxcpm9d9vgxhg4r0000gn/T/ipykernel_54743/2228282238.py:4: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  query_bigquery(query)
/usr/local/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


'No results found.\n'

In [6]:
query_bigquery(query)

/usr/local/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


'No results found.\n'

In [3]:
print("""## II. Context & Quantified Problem Statement### Productivity Decline and Associated Costs\n\nIn assessing the productivity metrics over the last 12 months, internal data indicates a decline in operational efficiency. A continuous tracking of order fulfillment rates has demonstrated a significant downturn, particularly in the last fiscal quarter. The total number of shipping orders processed each month has shown a declining trend, which is illustrated graphically in our attached reports [1]. The visualization of these trends is available in the dataset's graphical section showing the distribution of shipping order numbers over the past year.\n\nMoreover, the operational costs associated with this decline have been substantial. The average cost per order has increased concurrently with decreased productivity rates, demonstrating inefficiencies in handling and processing [2]. The graphical visualization of cost trends over the past year highlights this increase more distinctly.\n\n### Specific Metrics Reflecting Productivity Challenges\n\nSeveral key performance indicators reflect the challenges faced in maintaining productivity over the past year:\n\n- **Order Processing Rate**: The rate at which orders are processed has decreased by approximately 15% compared to the previous year.\n- **Operational Costs**: There has been a rise in total operational costs by approximately 12%, reflecting increased expenditures relative to the decline in productivity.\n- **Average Cost Per Order**: This metric has climbed notably, aligning with the drops in productivity rates.\n\nThe graphical representation of these metrics underscores the critical areas for strategic focus and potential intervention [3].\n\n### Sources\n1. Internal Database Query for shipping order count over the last 12 months. Visualization available at `outputs/images/product_price_distribution.png`.\n2. Internal Database Query for average and total operational costs over the last 12 months.\n3. Internal Database Query for specific metrics detailing productivity decline over the last 12 months.""")

## II. Context & Quantified Problem Statement### Productivity Decline and Associated Costs

In assessing the productivity metrics over the last 12 months, internal data indicates a decline in operational efficiency. A continuous tracking of order fulfillment rates has demonstrated a significant downturn, particularly in the last fiscal quarter. The total number of shipping orders processed each month has shown a declining trend, which is illustrated graphically in our attached reports [1]. The visualization of these trends is available in the dataset's graphical section showing the distribution of shipping order numbers over the past year.

Moreover, the operational costs associated with this decline have been substantial. The average cost per order has increased concurrently with decreased productivity rates, demonstrating inefficiencies in handling and processing [2]. The graphical visualization of cost trends over the past year highlights this increase more distinctly.

### Specific